In [1]:
import sys
print("Python executable:", sys.executable)
import networkx as nx
print("networkx version:", nx.__version__)

Python executable: /media/paulina/TOSHIBA EXT/2024/Codigos/MercadoLibre/venv/bin/python
networkx version: 3.5


# Red principal: co-ofertas Producto–Producto

Nodo = DOMAIN_ID (producto).

Arista = dos productos que participaron juntos en la misma oferta/evento.

Pesos de la arista:

*     cooffers: cuántas veces aparecieron juntos. --> Frecuencia de co-participación

*     joint_qty: ventas conjuntas asociadas a ese par. --> Intensidad comercial conjunta

In [2]:
import pandas as pd
from itertools import combinations
import networkx as nx

df = pd.read_csv("/media/paulina/TOSHIBA EXT/2024/Codigos/MercadoLibre/arquivos/ofertas_relampago.csv", encoding="latin1")

df["OFFER_START_DTTM"] = pd.to_datetime(df["OFFER_START_DTTM"])
df["OFFER_FINISH_DTTM"] = pd.to_datetime(df["OFFER_FINISH_DTTM"])

#ID de evento de oferta
df["EVENT_ID"] = df["OFFER_START_DTTM"]

# Rellenar NaN de SOLD_QUANTITY con 0 para no romper los cálculos
df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

In [ ]:
def crear_df_coofertas_productos(df, event_cols=None):
    """
    Crea un DataFrame de edges Producto–Producto basado en co-ofertas.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame original de ofertas relámpago.
    event_cols : list[str], opcional
        Columnas que definen un "evento de oferta".
        Por defecto: ["OFFER_START_DTTM", "OFFER_TYPE"].

    Devuelve
    --------
    edges_prod : pd.DataFrame
        DataFrame con columnas:
        - prod_i
        - prod_j
        - cooffers   (nº de eventos compartidos)  <--
        - joint_qty  (ventas conjuntas acumuladas) <--
    """
    if event_cols is None:
        event_cols = ["OFFER_START_DTTM", "OFFER_TYPE"]

    # Asegurar tipos de fecha en las columnas de evento
    for col in event_cols:
        if "DTTM" in col or "DATE" in col:
            df[col] = pd.to_datetime(df[col])

    # Asegurar que SOLD_QUANTITY no tenga NaN
    df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

    # Nos quedamos solo con lo necesario
    cols = event_cols + ["DOMAIN_ID", "SOLD_QUANTITY"]
    df_event_prod = df[cols].copy()

    edge_rows = []

    # Agrupar por evento (combinación de columnas event_cols)
    for _, group in df_event_prod.groupby(event_cols):
        # Agregamos ventas por producto dentro del evento
        prod_event = (
            group.groupby("DOMAIN_ID")["SOLD_QUANTITY"]
                 .sum()
                 .reset_index()
        )

        products = prod_event["DOMAIN_ID"].values
        qty_by_prod = prod_event.set_index("DOMAIN_ID")["SOLD_QUANTITY"]

        # Si solo hay un producto en el evento, no genera aristas
        if len(products) < 2:
            continue

        # Todas las combinaciones de productos dentro del evento
        for a, b in combinations(products, 2):
            # ventas conjuntas en ese evento: ventas(A) + ventas(B)
            joint_qty = float(qty_by_prod[a] + qty_by_prod[b])
            edge_rows.append((a, b, 1, joint_qty))

    edges_prod = (
        pd.DataFrame(edge_rows, columns=["prod_i", "prod_j", "cooffers", "joint_qty"])
        .groupby(["prod_i", "prod_j"], as_index=False)
        .agg(
            cooffers=("cooffers", "sum"),
            joint_qty=("joint_qty", "sum")
        )
    )

    return edges_prod

edges_prod = crear_df_coofertas_productos(df)
#edges_prod.to_csv("edges_cooffers_PRODUCTO_PRODUCTO.csv", index=False)

In [5]:
edges_prod

,prod_i,prod_j,cooffers,joint_qty
0,MLM-3D_PENS,MLM-ACTION_FIGURES,1,0.0
1,MLM-3D_PENS,MLM-ADHESIVE_TAPES,1,0.0
2,MLM-3D_PENS,MLM-AEROBICS_AND_FITNESS_EQUIPMENT,1,0.0
3,MLM-3D_PENS,MLM-ALL_TERRAIN_VEHICLE_TIRES,1,0.0
4,MLM-3D_PENS,MLM-ANKLE_AND_WRIST_WEIGHTS,1,4.0
...,...,...,...,...
178839,MLM-WORK_SCRUBS,MLM-WRISTWATCH_SCREEN_PROTECTORS,17,62.0
178840,MLM-WRENCHES,MLM-WRISTWATCHES,1,13.0
178841,MLM-WRENCH_SETS,MLM-WRISTWATCHES,1,10.0
178842,MLM-WRISTWATCHES,MLM-WRISTWATCH_SCREEN_PROTECTORS,30,210.0


In [6]:
def crear_df_nodos_productos(df):
    """
    Crea un DataFrame de nodos (productos) con atributos agregados.

    Devuelve columnas como:
    - DOMAIN_ID
    - vertical
    - dom_agg1
    - mean_stock, total_stock
    - mean_sold, total_sold
    - stockout_rate
    """
    df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

    nodos_prod = (
        df.groupby("DOMAIN_ID")
          .agg(
              vertical=("VERTICAL", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
              dom_agg1=("DOM_DOMAIN_AGG1", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
              mean_stock=("INVOLVED_STOCK", "mean"),
              total_stock=("INVOLVED_STOCK", "sum"),
              mean_sold=("SOLD_QUANTITY", "mean"),
              total_sold=("SOLD_QUANTITY", "sum"),
              stockout_rate=("REMAINING_STOCK_AFTER_END", lambda x: (x < 0).mean())
          )
          .reset_index()
    )

    return nodos_prod

nodos_prod = crear_df_nodos_productos(df)
#nodos_prod.to_csv("nodos_PRODUCTO.csv", index=False)

In [7]:
nodos_prod

,DOMAIN_ID,vertical,dom_agg1,mean_stock,total_stock,mean_sold,total_sold,stockout_rate
0,MLM-3D_PENS,CE,COMPUTERS,15.000000,15,0.000000,0.0,0.000000
1,MLM-3D_PRINTERS,CE,COMPUTERS,5.000000,5,0.000000,0.0,0.000000
2,MLM-3D_PRINTER_FILAMENTS,CE,COMPUTERS,5.000000,5,0.000000,0.0,0.000000
3,MLM-3D_PRINTER_NOZZLES,CE,COMPUTERS,10.000000,10,0.000000,0.0,0.000000
4,MLM-ABDOMINAL_TONING_BELTS,BEAUTY & HEALTH,BEAUTY EQUIPMENT,15.000000,15,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...
1261,MLM-WRENCH_SETS,HOME & INDUSTRY,TOOLS AND CONSTRUCTION,8.500000,17,5.000000,10.0,0.000000
1262,MLM-WRISTWATCHES,APP & SPORTS,APPAREL ACCESORIES,24.400463,10541,1.430556,618.0,0.011574
1263,MLM-WRISTWATCH_SCREEN_PROTECTORS,CE,COMPUTERS,19.042254,1352,1.084507,77.0,0.000000
1264,MLM-XYLOPHONES_AND_METALLOPHONES,ENTERTAINMENT,MUSICAL INSTRUMENTS,8.000000,8,0.000000,0.0,0.000000


In [8]:
def crear_red_coofertas_productos(edges_prod, nodos_prod):
    """
    Crea la red Producto–Producto a partir de:
      - edges_prod: DataFrame de aristas con cooffers y joint_qty
      - nodos_prod: DataFrame con atributos de cada DOMAIN_ID

    Retorna:
      - G_prod: networkx.Graph listo para exportar a Gephi.
    """
    G_prod = nx.Graph()

    # Añadir nodos con atributos
    for _, row in nodos_prod.iterrows():
        G_prod.add_node(
            row["DOMAIN_ID"],
            vertical=row["vertical"],
            dom_agg1=row["dom_agg1"],
            mean_stock=float(row["mean_stock"]) if pd.notna(row["mean_stock"]) else None,
            total_stock=float(row["total_stock"]) if pd.notna(row["total_stock"]) else None,
            mean_sold=float(row["mean_sold"]) if pd.notna(row["mean_sold"]) else None,
            total_sold=float(row["total_sold"]) if pd.notna(row["total_sold"]) else None,
            stockout_rate=float(row["stockout_rate"]) if pd.notna(row["stockout_rate"]) else None,
        )

    # Añadir aristas con pesos
    for _, row in edges_prod.iterrows():
        G_prod.add_edge(
            row["prod_i"],
            row["prod_j"],
            cooffers=int(row["cooffers"]),      # nº de eventos que comparten
            joint_qty=float(row["joint_qty"])  # ventas conjuntas acumuladas
        )

    return G_prod

G_prod = crear_red_coofertas_productos(edges_prod, nodos_prod)

print(G_prod.number_of_nodes(), "nodos")
print(G_prod.number_of_edges(), "aristas")

1266 nodos
178844 aristas


In [9]:
nx.write_gexf(G_prod, "Red_cooffers_PRODUCTO_PRODUCTO.gexf")